# InMAP-Canada Tutorial
This workbook will walk through some manipulations of the InMAP Canada input and output files

## Emissions - Loading, Displaying and Modifying files
Emissions files are in shape (.shp) format, and can be displayed and manipulated with any GIS software or geospatial code package. We will be using Geopandas

In [1]:
import geopandas as gpd 
import numpy as np #matrix functions
import inmap_funcs as ihf #these are functions we wrote to help process data. 

In [ ]:
#Set the file paths for the emissions.
fpath = '/home/tfmrodge/scratch/GEMMACH_data/data/Inmap_outputs/Emissions/Emissions_shp/Diff/'#Beluga
ptspath = '/home/tfmrodge/scratch/GEMMACH_data/data/Inmap_outputs/Emissions/Emissions_shp/'#Beluga
figpath='/home/tfmrodge/scratch/GEMMACH_data/Figs/20240306/'
#List of scenarios with emissions and output naming schemes, for convenience
scenarios={1:['BASEGM_2015_017','BASEGM_2015_017'],2:['BAU_2020_E108','BAU_2020_105'],3:['BAU_2025_E107','BAU_2025_106'],4:['BAU_2030_E107','BAU_2030_106'],
5:['BAU_2035_E107','BAU_2035_106'],6:['BPT_2015_E002','BPT_2015_002'],7:['BPT_2015_E006','BPT_2015_006'],
8:['BPT_2015_E015','BPT_2015_015'],9:['COVID_2020_E003','COVID_2020_003'],10:['BPT_2015_E016','BPT_2015_016']}
#List the pollutants to display
listvals=['NH3','NOx','PM25','SOx','VOC']
loadfile=True #False
dopts=False #This flag says if you have point or gridded major emissions.
sumemissions=True
crs='ESRI:102002'
#Next, we load the files
sID = 1 #define which one of the scenarios from dict
scenario = scenarios[sID][0]
#Load in the area emissions
areaemissions = gpd.read_file(fpath+scenario+'_area'+suffix).to_crs(crs)
#Load the major emissions. Note that the points files are slow!
majoremissions=gpd.read_file(fpath+scenario+'_major'+suffix).to_crs(crs)
#Add a sum of emissions if major emissions are gridded.
allemissions = areaemissions.copy(deep=True) 
if dopts:
    pass
else:
    allemissions.loc[:,listvals]=allemissions.loc[:,listvals]+majoremissions.loc[:,listvals]
print('loaded and summed for '+scenario)

In [ ]:
#Next, use the plotting function that we have defined to plot the emissions.
emisslist=[areaemissions, majoremissions,allemissions] #area, major, combined
#most of the options here are cosmetic, see the function in inmap_funcs for full details
fig = ihf.plot_emissions(emisslist,provinces,legend=True,lgdshk = 0.3,lnwdth = 0.05,
                     alpha = 1.0,cmap=cmap,listvals=listvals,figpath=figpath,scenario=scenario+'_diff',diff=True,dopts=dopts)
fig.show()
print('plotted '+scenario)

In [ ]:
#If we want to modify the emissions, we need to manipulate the inputs and then save again
#Let's say we want to drop NOx by 10% everywhere:
areaemissions.loc[:,'NOx'] = 0.9*areaemissions.loc[:,'NOx']
#Then we export with geopandas.
areaemissions.to_file(fpath+scenario+'_area'+suffix[:-4]+'_90pct.shp')
majoremissions.loc[:,'NOx'] = 0.9*majoremissions.loc[:,'NOx']
majoremissions.to_file(fpath+scenario+'_major'+suffix[:-4]+'_90pct.shp')
#Then, you can modify the .toml file when you run next to use this scenario
print(fpath+scenario+'_area'+suffix[:-4]+'_90pct.shp ,'+fpath+scenario+'_major'+suffix[:-4]+'_90pct.shp')

## InMAP Outputs - Loading, summary stats, displaying files
Output files are likewise in shape (.shp) format, and can be displayed and manipulated with any GIS software or geospatial code package.


In [ ]:
#Plot Concentrations vs GEMMACH and generate summary statistics
fpath = '/home/tfmrodge/scratch/GEMMACH_data/data/Inmap_outputs/Outputs/'#Beluga
figpath='/home/tfmrodge/scratch/GEMMACH_data/Figs/
#List pairs of base and output emissions
pairs=[['BasePM25','TotalPM25'],['BasePNO3','PNO3'],['BasePNH4',
        'PNH4'],['BasePSO4','PSO4'],['BaseSOA','SOA'],['BasePrimPM25','PrimPM25']]
#Stats that will be calculated. This is the full list, can modify if you want fewer.
stats = ['RMSE', 'MeanBias', 'MeanError', 'MeanFractionalBias', 
        'MeanFractionalError', 'ModelRatio', 'Regression']
#Regions for summary statistics - in this case by province
geoareas = ['Canada','British Columbia', 'Quebec', 'Nunavut', 'Prince Edward Island',
 'Saskatchewan', 'Yukon', 'Manitoba', 'Ontario', 'New Brunswick', 'Northwest Territories',
 'Alberta', 'Newfoundland and Labrador', 'Nova Scotia']
prefix='20240305_InmapOuts_'
suffix='_diff_combined.shp'
#Define the ID of the scenario
sIDs = [1]
scenario = scenarios[sID][0]#Set the scenario name 
calcdelta=True #Flag to calculate the difference between GEMMACH and InMAp
loadfile=True #Flag to load the file (vs using previously loaded)
cmap=matplotlib.cm.RdBu_r
#Extent to be mapped
xylims=[(-2579201.070414297,3165870.),(76856.48815160134,4270028.)] #All of Canda
#xylims=[(-2200000,-1300000),(1190000,1800000)] #BC 
#xylims=[(1088000,1900000),(332000,1250000)] #ON-QC Corridor
if loadfile:
    inmap_outs = ihf.load_inmap(fpath+prefix+scenario+suffix,basefile=None,crs='ESRI:102002',clipped=False)
    #Add provinces to get summary statistics
    inmap_outs = sjoin(inmap_outs, provinces.loc[:,['PRENAME','geometry']], how='left',predicate='intersects')
#'Base' data is the GEMMACH output. 
if calcdelta:
    for vals in [['BasePM25','TotalPM25'],['BasePNO3','PNO3'],['BasePNH4',
                      'PNH4'],['BasePSO4','PSO4'],['BaseSOA','SOA'],['BasePrimPM25','PrimPM25']]:
        delname = 'delta_'+vals[0][4:]
        if vals[0] == 'BasePrimPM25':
            #Need to calculate out primary PM2.5 in the base scenario
            inmap_outs.loc[:,vals[0]] = (inmap_outs.BasePM25) - (inmap_outs.loc[:,['BasePNO3','BasePNH4','BasePSO4','BaseSOA']].sum(axis=1))
        inmap_outs.loc[:,delname] = inmap_outs.loc[:,vals[1]] - inmap_outs.loc[:,vals[0]] 

In [ ]:
fig = ihf.plot_pollutants(inmap_outs,provinces,legend=True,lgdshk = 0.3,lnwdth = 0.05,alpha = 1.0,
                              cmap=cmap,listvals=pairs,figpath=figpath,scenario=scenario+'_'+prefix[:8],diff=True,xylims=xylims)
fig.show()

In [ ]:
#statdf function - calculates summary stats for different pairs of values in different regions
areastatdf = ihf.summstats(inmap_outs,pairs,stats,geoareas,popwt=None,geoname='PRENAME',popcol='TotalPop')
areastatdf.to_csv(figpath+scenario+'_areasummstats.csv')
popstatdf = ihf.summstats(inmap_outs,pairs,stats,geoareas,popwt=True,geoname='PRENAME',popcol='TotalPop')
popstatdf.to_csv(figpath+scenario+'_popsummstats.csv')